# 01 — Data Loading and Preprocessing (SoccerMon 2020)

This notebook reconstructs the 2020 analysis resource used by the downstream landmark modelling notebooks.

## Scientific scope

SoccerMon provides minute-resolution monitoring data, while injury supervision is available at the athlete-day/session level and does **not** provide an exact within-session injury-onset timestamp. Accordingly, this notebook does not create minute-specific injury labels or prospective within-session injury-risk targets.

The preprocessing objective is to construct:

1. a cleaned minute-resolution athlete-session table; and
2. a one-row-per-athlete-session master table containing session summaries, temporally aligned contextual variables, and the session-level injury-associated indicator.

A positive label means that an injury report is associated with the same athlete and calendar date as an eligible exposure session. It should not be interpreted as a known injury-onset minute or as proof that every observation in that session preceded biological injury onset.

## Session reconstruction

Minute-level observations are grouped by team and calendar date. A gap greater than 30 minutes identifies a separate temporal block. Team-days containing more than one block are excluded from the primary resource because the available injury report cannot identify which within-day block is associated with the injury report. Eligible single-block sessions must span at least 30 minutes.

## Reproducibility

Raw-data locations are configured through the `SOCCERMON_ROOT` environment variable. If it is not set, the notebook expects the raw dataset under `<repository>/data`.

Generated files are written to `<repository>/results/processed_data`.


In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## 1. Repository and data paths

No user-specific absolute paths are embedded in this notebook.

Set `SOCCERMON_ROOT` to the directory containing the SoccerMon `subjective/` folder and the `2020/` minute-level data tree when the raw data are stored outside the repository.


In [ ]:
def find_project_root(start: Path) -> Path:
    """Return the repository root when executed from the root or notebooks/ directory."""
    start = start.resolve()
    if start.name.lower() == "notebooks":
        return start.parent
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_ROOT = Path(
    os.environ.get("SOCCERMON_ROOT", PROJECT_ROOT / "data")
).expanduser().resolve()

SUBJECTIVE_DIR = DATA_ROOT / "subjective"
MINUTE_2020_DIR = DATA_ROOT / "2020"
OUTPUT_DIR = PROJECT_ROOT / "results" / "processed_data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("SoccerMon root:", DATA_ROOT)
print("Output directory:", OUTPUT_DIR)

if not SUBJECTIVE_DIR.exists():
    raise FileNotFoundError(
        "SoccerMon subjective-data directory was not found. "
        "Set the SOCCERMON_ROOT environment variable to the raw dataset root. "
        f"Expected: {SUBJECTIVE_DIR}"
    )

if not MINUTE_2020_DIR.exists():
    raise FileNotFoundError(
        "SoccerMon 2020 minute-data directory was not found. "
        "Set the SOCCERMON_ROOT environment variable to the raw dataset root. "
        f"Expected: {MINUTE_2020_DIR}"
    )


## 2. Helper functions

In [ ]:
def load_minute_file(path: Path) -> pd.DataFrame:
    """Load one minute-aggregate file and normalize identifiers and numeric fields."""
    df = pd.read_csv(path, low_memory=False)
    df.columns = [column.strip() for column in df.columns]

    df = df.rename(
        columns={
            "player_name_": "player_name",
            "date_": "date",
            "minute_": "minute",
            "minute_idx_": "minute_idx",
        }
    )

    id_cols = ["player_name", "date", "minute", "minute_idx"]
    missing = set(id_cols) - set(df.columns)
    if missing:
        raise ValueError(f"Missing required minute-file columns in {path}: {sorted(missing)}")

    df["player_name"] = df["player_name"].astype("string")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["minute"] = pd.to_datetime(df["minute"], errors="coerce")
    df["minute_idx"] = pd.to_numeric(
        df["minute_idx"], errors="coerce"
    ).astype("Int64")

    numeric_cols = [column for column in df.columns if column not in id_cols]
    df[numeric_cols] = df[numeric_cols].apply(
        pd.to_numeric, errors="coerce"
    )

    return df


def add_session_id(df: pd.DataFrame) -> pd.DataFrame:
    """Add team and deterministic team-date session identifiers."""
    out = df.copy()
    out["date"] = pd.to_datetime(out["date"], errors="coerce")
    out["team"] = (
        out["player_name"]
        .astype(str)
        .str.split("-", n=1)
        .str[0]
    )
    out["session_id"] = (
        out["team"].astype(str)
        + "_"
        + out["date"].dt.strftime("%Y-%m-%d")
    )
    return out


def safe_left_merge(
    base: pd.DataFrame,
    other: pd.DataFrame,
    cols: list[str],
    name: str,
) -> pd.DataFrame:
    """Perform a many-to-one left merge after asserting unique source keys."""
    other_small = other[cols].drop_duplicates()
    duplicate_count = other_small.duplicated(
        ["player_name", "session_id"]
    ).sum()

    assert duplicate_count == 0, (
        f"Duplicate player-session keys detected in {name}: {duplicate_count}"
    )

    return base.merge(
        other_small,
        on=["player_name", "session_id"],
        how="left",
        validate="m:1",
    )


def restrict_to_session_keys(
    df: pd.DataFrame,
    keys: pd.DataFrame,
) -> pd.DataFrame:
    """Restrict a table to athlete-session keys present in the modelling resource."""
    return df.merge(
        keys,
        on=["player_name", "session_id"],
        how="inner",
        validate="m:1",
    )


def lag_subjective(
    df: pd.DataFrame,
    value_cols: list[str],
    lag_days: int = 1,
) -> pd.DataFrame:
    """Map values recorded on day D to a session on day D + lag_days."""
    out = df.copy()
    out["date"] = (
        pd.to_datetime(out["date"], errors="coerce")
        + pd.to_timedelta(lag_days, unit="D")
    )
    out = add_session_id(out)
    return out[["player_name", "session_id"] + value_cols]


def melt_daily(df: pd.DataFrame, value_name: str) -> pd.DataFrame:
    """Convert a SoccerMon wide daily table to long format with explicit DD.MM.YYYY parsing."""
    df = df.copy()

    if "Date" in df.columns:
        date_col = "Date"
    elif "date" in df.columns:
        date_col = "date"
    else:
        raise ValueError(
            f"No date column found in dataframe for {value_name}"
        )

    out = df.melt(
        id_vars=date_col,
        var_name="player_name",
        value_name=value_name,
    )

    out = out.rename(columns={date_col: "date"})
    out["date"] = pd.to_datetime(
        out["date"],
        format="%d.%m.%Y",
        errors="coerce",
    )

    return out


def lag_workload(
    df: pd.DataFrame,
    value_col: str,
    lag_days: int = 1,
) -> pd.DataFrame:
    """Map a workload value recorded on day D to day D + lag_days."""
    temp = df[["player_name", "date", value_col]].copy()
    temp["date"] = (
        pd.to_datetime(temp["date"], errors="coerce")
        + pd.Timedelta(days=lag_days)
    )
    return add_session_id(temp)


## 3. Load subjective, workload, wellness, injury, and session-level data

In [ ]:
relative_files = {
    "game_performance": Path("game-performance/game-performance.csv"),
    "illness": Path("illness/illness.csv"),
    "injury": Path("injury/injury.csv"),
    "acwr": Path("training-load/acwr.csv"),
    "atl": Path("training-load/atl.csv"),
    "ctl28": Path("training-load/ctl28.csv"),
    "ctl42": Path("training-load/ctl42.csv"),
    "daily_load": Path("training-load/daily_load.csv"),
    "monotony": Path("training-load/monotony.csv"),
    "strain": Path("training-load/strain.csv"),
    "weekly_load": Path("training-load/weekly_load.csv"),
    "session": Path("training-load/session.json"),
    "fatigue": Path("wellness/fatigue.csv"),
    "mood": Path("wellness/mood.csv"),
    "readiness": Path("wellness/readiness.csv"),
    "sleep_duration": Path("wellness/sleep_duration.csv"),
    "sleep_quality": Path("wellness/sleep_quality.csv"),
    "soreness": Path("wellness/soreness.csv"),
    "stress": Path("wellness/stress.csv"),
}

missing_files = [
    SUBJECTIVE_DIR / relative_path
    for relative_path in relative_files.values()
    if not (SUBJECTIVE_DIR / relative_path).exists()
]

if missing_files:
    formatted = "\n".join(f"  - {path}" for path in missing_files)
    raise FileNotFoundError(
        "Required SoccerMon input files are missing:\n" + formatted
    )


def read_subjective_csv(name: str) -> pd.DataFrame:
    return pd.read_csv(
        SUBJECTIVE_DIR / relative_files[name],
        low_memory=False,
    )


game_performance_data = read_subjective_csv("game_performance")
illness_data = read_subjective_csv("illness")
injury_data = read_subjective_csv("injury")

acwr_data = read_subjective_csv("acwr")
atl_data = read_subjective_csv("atl")
ctl28_data = read_subjective_csv("ctl28")
ctl42_data = read_subjective_csv("ctl42")
daily_load_data = read_subjective_csv("daily_load")
monotony_data = read_subjective_csv("monotony")
strain_data = read_subjective_csv("strain")
weekly_load_data = read_subjective_csv("weekly_load")

fatigue_data = read_subjective_csv("fatigue")
mood_data = read_subjective_csv("mood")
readiness_data = read_subjective_csv("readiness")
sleep_duration_data = read_subjective_csv("sleep_duration")
sleep_quality_data = read_subjective_csv("sleep_quality")
soreness_data = read_subjective_csv("soreness")
stress_data = read_subjective_csv("stress")

with open(
    SUBJECTIVE_DIR / relative_files["session"],
    "r",
    encoding="utf-8",
) as file:
    session_dict = json.load(file)

session_rows = []
for player_name, sessions in session_dict.items():
    for session in sessions:
        session_rows.append(
            {
                "player_name": player_name,
                "date": pd.to_datetime(
                    session["date"],
                    format="%d.%m.%Y",
                    errors="raise",
                ),
                "srpe": session["srpe"],
                "rpe": session["rpe"],
                "duration": session["duration"],
            }
        )

session_data = pd.DataFrame(session_rows)

print("Subjective/session inputs loaded successfully.")
print("Session JSON rows:", len(session_data))


## 4. Normalize daily tables

Daily wide tables use explicit day-month-year parsing. This explicit format is required for reproducibility because generic date parsing can silently reinterpret ambiguous dates.

Wellness variables other than sleep are shifted by one day before being attached to an exposure session. Sleep variables are retained on the same calendar date as descriptors of the preceding night's sleep; the dataset does not provide a verified pre-session submission timestamp, so downstream PRE analyses must retain that limitation.

Session-derived `srpe`, `rpe`, and `duration` summaries are retained in the master resource for descriptive/session accounting. They are not treated here as guaranteed pre-session predictors.


In [ ]:
wellness_date_renames = {
    "fatigue": (fatigue_data, "Fatigue Data"),
    "mood": (mood_data, "Mood Data"),
    "readiness": (readiness_data, "Readiness Data"),
    "sleep_duration": (sleep_duration_data, "SleepDurH Data"),
    "sleep_quality": (sleep_quality_data, "SleepQuality Data"),
    "soreness": (soreness_data, "Soreness Data"),
}

for _, (frame, original_date_column) in wellness_date_renames.items():
    if original_date_column in frame.columns:
        frame.rename(
            columns={original_date_column: "Date"},
            inplace=True,
        )

acwr_data = melt_daily(acwr_data, "acwr")
atl_data = melt_daily(atl_data, "atl")
ctl28_data = melt_daily(ctl28_data, "ctl28")
ctl42_data = melt_daily(ctl42_data, "ctl42")
daily_load_data = melt_daily(daily_load_data, "daily_load")
weekly_load_data = melt_daily(weekly_load_data, "weekly_load")
monotony_data = melt_daily(monotony_data, "monotony")
strain_data = melt_daily(strain_data, "strain")

fatigue_data = melt_daily(fatigue_data, "fatigue")
mood_data = melt_daily(mood_data, "mood")
readiness_data = melt_daily(readiness_data, "readiness")
sleep_duration_data = melt_daily(
    sleep_duration_data,
    "sleep_duration",
)
sleep_quality_data = melt_daily(
    sleep_quality_data,
    "sleep_quality",
)
soreness_data = melt_daily(soreness_data, "soreness")
stress_data = melt_daily(stress_data, "stress")

game_performance_data = game_performance_data.rename(
    columns={"timestamp": "date"}
)
illness_data = illness_data.rename(columns={"timestamp": "date"})
injury_data = injury_data.rename(columns={"timestamp": "date"})

for label_name, frame in {
    "injury": injury_data,
    "illness": illness_data,
}.items():
    required = {"player_name", "date"}
    missing = required - set(frame.columns)
    assert not missing, (
        f"Missing required columns in {label_name} table: {sorted(missing)}"
    )

print("Daily tables normalized successfully.")


## 5. Load and reconstruct minute-level exposure sessions

In [ ]:
minute_files = sorted(
    MINUTE_2020_DIR.rglob("minute_aggregates.csv")
)

print("Minute aggregate files found:", len(minute_files))

if not minute_files:
    raise FileNotFoundError(
        "No minute_aggregates.csv files were found under "
        f"{MINUTE_2020_DIR}"
    )

minute_frames = [
    load_minute_file(path)
    for path in minute_files
]

minute_2020_data = pd.concat(
    minute_frames,
    ignore_index=True,
)

minute_2020_data["date"] = pd.to_datetime(
    minute_2020_data["date"],
    errors="coerce",
).dt.date

minute_2020_data["minute"] = pd.to_datetime(
    minute_2020_data["minute"],
    errors="coerce",
)

minute_2020_data = (
    minute_2020_data.loc[
        minute_2020_data["player_name"].notna()
        & minute_2020_data["date"].notna()
        & minute_2020_data["minute"].notna()
    ]
    .copy()
    .reset_index(drop=True)
)

minute_2020_data["team"] = (
    minute_2020_data["player_name"]
    .astype(str)
    .str.split("-", n=1)
    .str[0]
)

minute_2020_data = (
    minute_2020_data
    .sort_values(["team", "date", "minute"])
    .reset_index(drop=True)
)

minute_2020_data["gap_minutes"] = (
    minute_2020_data
    .groupby(["team", "date"])["minute"]
    .diff()
    .dt.total_seconds()
    .div(60)
)

GAP_THRESHOLD_MIN = 30

minute_2020_data["new_session_block"] = (
    minute_2020_data["gap_minutes"].isna()
    | (minute_2020_data["gap_minutes"] > GAP_THRESHOLD_MIN)
)

minute_2020_data["session_block"] = (
    minute_2020_data
    .groupby(["team", "date"])["new_session_block"]
    .cumsum()
    .astype(int)
)

blocks_per_day = (
    minute_2020_data
    .groupby(["team", "date"])["session_block"]
    .nunique()
    .rename("n_session_blocks")
    .reset_index()
)

single_session_days = blocks_per_day.loc[
    blocks_per_day["n_session_blocks"] == 1,
    ["team", "date"],
]

minute_2020_data = (
    minute_2020_data
    .merge(
        single_session_days,
        on=["team", "date"],
        how="inner",
        validate="m:1",
    )
    .reset_index(drop=True)
)

minute_2020_data["session_id"] = (
    minute_2020_data["team"].astype(str)
    + "_"
    + minute_2020_data["date"].astype(str)
)

session_duration = (
    minute_2020_data
    .groupby(["team", "date", "session_id"])
    .agg(
        start_time=("minute", "min"),
        end_time=("minute", "max"),
    )
    .reset_index()
)

session_duration["duration_minutes"] = (
    session_duration["end_time"]
    - session_duration["start_time"]
).dt.total_seconds() / 60

MIN_SESSION_MINUTES = 30

valid_sessions = session_duration.loc[
    session_duration["duration_minutes"] >= MIN_SESSION_MINUTES,
    ["team", "date", "session_id"],
]

minute_2020_data = (
    minute_2020_data
    .merge(
        valid_sessions,
        on=["team", "date", "session_id"],
        how="inner",
        validate="m:1",
    )
    .reset_index(drop=True)
)

duplicate_key = ["player_name", "session_id", "minute"]
n_before = len(minute_2020_data)
n_duplicate_rows = int(
    minute_2020_data.duplicated(
        subset=duplicate_key,
        keep="first",
    ).sum()
)

if n_duplicate_rows:
    minute_2020_data = (
        minute_2020_data
        .sort_values(duplicate_key)
        .drop_duplicates(
            subset=duplicate_key,
            keep="first",
        )
        .reset_index(drop=True)
    )

assert not minute_2020_data.duplicated(
    subset=duplicate_key
).any()

minute_2020_data = minute_2020_data.drop(
    columns=[
        "gap_minutes",
        "new_session_block",
        "session_block",
    ],
    errors="ignore",
)

print("Rows before deduplication:", n_before)
print("Duplicate rows removed:", n_before - len(minute_2020_data))
print("Eligible team-days:", minute_2020_data["session_id"].nunique())
print("Minute observations:", len(minute_2020_data))
print("Minute-level player-session-time keys are unique.")


## 6. Construct athlete-session summaries and temporally aligned contextual tables

In [ ]:
minute_session = (
    minute_2020_data
    .groupby(
        ["player_name", "team", "session_id"],
        as_index=False,
    )
    .agg(
        total_minutes=("minute_idx", "size"),
        speed_mean_sess=("speed_mean", "mean"),
        speed_sum_sess=("speed_mean", "sum"),
        hr_mean_sess=("heart_rate_mean", "mean"),
        hr_sum_sess=("heart_rate_mean", "sum"),
        inst_acc_mean_sess=("inst_acc_impulse_mean", "mean"),
        inst_acc_sum_sess=("inst_acc_impulse_mean", "sum"),
        hacc_mean_sess=("hacc_mean", "mean"),
        hacc_sum_sess=("hacc_mean", "sum"),
        accl_x_mean_sess=("accl_x_mean", "mean"),
        accl_x_sum_sess=("accl_x_mean", "sum"),
        accl_y_mean_sess=("accl_y_mean", "mean"),
        accl_y_sum_sess=("accl_y_mean", "sum"),
        accl_z_mean_sess=("accl_z_mean", "mean"),
        accl_z_sum_sess=("accl_z_mean", "sum"),
        gyro_x_mean_sess=("gyro_x_mean", "mean"),
        gyro_x_sum_sess=("gyro_x_mean", "sum"),
        gyro_y_mean_sess=("gyro_y_mean", "mean"),
        gyro_y_sum_sess=("gyro_y_mean", "sum"),
        gyro_z_mean_sess=("gyro_z_mean", "mean"),
        gyro_z_sum_sess=("gyro_z_mean", "sum"),
    )
)

session_daily = (
    session_data
    .assign(
        date=pd.to_datetime(
            session_data["date"],
            errors="coerce",
        )
    )
    .groupby(["player_name", "date"], as_index=False)
    .agg(
        srpe_sum=("srpe", "sum"),
        duration_sum=("duration", "sum"),
        rpe_mean=("rpe", "mean"),
        rpe_max=("rpe", "max"),
        n_sessions=("rpe", "count"),
    )
    .pipe(add_session_id)
)

workload_long = {
    "acwr": add_session_id(acwr_data),
    "atl": add_session_id(atl_data),
    "ctl28": add_session_id(ctl28_data),
    "ctl42": add_session_id(ctl42_data),
    "daily_load": add_session_id(daily_load_data),
    "monotony": add_session_id(monotony_data),
    "strain": add_session_id(strain_data),
    "weekly_load": add_session_id(weekly_load_data),
}

workload_lagged = {
    name: lag_workload(frame, name, lag_days=1)
    for name, frame in workload_long.items()
}

sleep_duration_session = add_session_id(
    sleep_duration_data
)
sleep_quality_session = add_session_id(
    sleep_quality_data
)

injury_daily = (
    injury_data
    .assign(
        date=pd.to_datetime(
            injury_data["date"],
            dayfirst=True,
            errors="coerce",
        ),
        injury=1,
    )
    .groupby(["player_name", "date"], as_index=False)["injury"]
    .max()
    .pipe(add_session_id)
)

illness_daily = (
    illness_data
    .assign(
        date=pd.to_datetime(
            illness_data["date"],
            dayfirst=True,
            errors="coerce",
        ),
        illness=1,
    )
    .groupby(["player_name", "date"], as_index=False)["illness"]
    .max()
    .pipe(add_session_id)
)

fatigue_lag = lag_subjective(
    fatigue_data[["player_name", "date", "fatigue"]],
    ["fatigue"],
    lag_days=1,
)
mood_lag = lag_subjective(
    mood_data[["player_name", "date", "mood"]],
    ["mood"],
    lag_days=1,
)
readiness_lag = lag_subjective(
    readiness_data[["player_name", "date", "readiness"]],
    ["readiness"],
    lag_days=1,
)
soreness_lag = lag_subjective(
    soreness_data[["player_name", "date", "soreness"]],
    ["soreness"],
    lag_days=1,
)
stress_lag = lag_subjective(
    stress_data[["player_name", "date", "stress"]],
    ["stress"],
    lag_days=1,
)

print("Athlete-session summaries:", len(minute_session))
print("Temporal contextual tables constructed successfully.")


## 7. Restrict contextual data to eligible athlete-session keys

Only athlete-session keys present in the reconstructed minute-level resource are retained before the master table is assembled.


In [ ]:
session_keys = (
    minute_session[["player_name", "session_id"]]
    .drop_duplicates()
)

session_daily_g = restrict_to_session_keys(
    session_daily,
    session_keys,
)

atl_g = restrict_to_session_keys(
    workload_lagged["atl"][["player_name", "session_id", "atl"]],
    session_keys,
)
ctl28_g = restrict_to_session_keys(
    workload_lagged["ctl28"][["player_name", "session_id", "ctl28"]],
    session_keys,
)
ctl42_g = restrict_to_session_keys(
    workload_lagged["ctl42"][["player_name", "session_id", "ctl42"]],
    session_keys,
)
daily_load_g = restrict_to_session_keys(
    workload_lagged["daily_load"][
        ["player_name", "session_id", "daily_load"]
    ],
    session_keys,
)
weekly_load_g = restrict_to_session_keys(
    workload_lagged["weekly_load"][
        ["player_name", "session_id", "weekly_load"]
    ],
    session_keys,
)
acwr_g = restrict_to_session_keys(
    workload_lagged["acwr"][["player_name", "session_id", "acwr"]],
    session_keys,
)
monotony_g = restrict_to_session_keys(
    workload_lagged["monotony"][
        ["player_name", "session_id", "monotony"]
    ],
    session_keys,
)
strain_g = restrict_to_session_keys(
    workload_lagged["strain"][
        ["player_name", "session_id", "strain"]
    ],
    session_keys,
)

sleep_dur_g = restrict_to_session_keys(
    sleep_duration_session[
        ["player_name", "session_id", "sleep_duration"]
    ],
    session_keys,
)
sleep_q_g = restrict_to_session_keys(
    sleep_quality_session[
        ["player_name", "session_id", "sleep_quality"]
    ],
    session_keys,
)

fatigue_g = restrict_to_session_keys(fatigue_lag, session_keys)
mood_g = restrict_to_session_keys(mood_lag, session_keys)
readiness_g = restrict_to_session_keys(readiness_lag, session_keys)
soreness_g = restrict_to_session_keys(soreness_lag, session_keys)
stress_g = restrict_to_session_keys(stress_lag, session_keys)

injury_g = restrict_to_session_keys(
    injury_daily[["player_name", "session_id", "injury"]],
    session_keys,
)
illness_g = restrict_to_session_keys(
    illness_daily[["player_name", "session_id", "illness"]],
    session_keys,
)

coverage_tables = {
    "atl": atl_g,
    "ctl28": ctl28_g,
    "ctl42": ctl42_g,
    "daily_load": daily_load_g,
    "weekly_load": weekly_load_g,
    "acwr": acwr_g,
    "monotony": monotony_g,
    "strain": strain_g,
}

print("CONTEXTUAL COVERAGE ON ELIGIBLE ATHLETE-SESSIONS")
print("=" * 60)
print(f"Total athlete-session keys: {len(session_keys):,}")

for name, frame in coverage_tables.items():
    matched = frame[["player_name", "session_id"]].drop_duplicates()
    print(
        f"{name:12s}: {len(matched):,} matched athlete-sessions "
        f"({100 * len(matched) / len(session_keys):.1f}%)"
    )


## 8. Assemble the master athlete-session table

In [ ]:
master_session = safe_left_merge(
    minute_session,
    session_daily_g,
    [
        "player_name",
        "session_id",
        "srpe_sum",
        "duration_sum",
        "rpe_mean",
        "rpe_max",
        "n_sessions",
    ],
    "session_daily",
)

merge_specs = [
    (atl_g, ["player_name", "session_id", "atl"], "atl"),
    (ctl28_g, ["player_name", "session_id", "ctl28"], "ctl28"),
    (ctl42_g, ["player_name", "session_id", "ctl42"], "ctl42"),
    (
        daily_load_g,
        ["player_name", "session_id", "daily_load"],
        "daily_load",
    ),
    (
        monotony_g,
        ["player_name", "session_id", "monotony"],
        "monotony",
    ),
    (strain_g, ["player_name", "session_id", "strain"], "strain"),
    (
        weekly_load_g,
        ["player_name", "session_id", "weekly_load"],
        "weekly_load",
    ),
    (acwr_g, ["player_name", "session_id", "acwr"], "acwr"),
    (
        sleep_dur_g,
        ["player_name", "session_id", "sleep_duration"],
        "sleep_duration",
    ),
    (
        sleep_q_g,
        ["player_name", "session_id", "sleep_quality"],
        "sleep_quality",
    ),
    (
        fatigue_g,
        ["player_name", "session_id", "fatigue"],
        "fatigue_lag1",
    ),
    (
        mood_g,
        ["player_name", "session_id", "mood"],
        "mood_lag1",
    ),
    (
        readiness_g,
        ["player_name", "session_id", "readiness"],
        "readiness_lag1",
    ),
    (
        soreness_g,
        ["player_name", "session_id", "soreness"],
        "soreness_lag1",
    ),
    (
        stress_g,
        ["player_name", "session_id", "stress"],
        "stress_lag1",
    ),
    (
        injury_g,
        ["player_name", "session_id", "injury"],
        "injury",
    ),
    (
        illness_g,
        ["player_name", "session_id", "illness"],
        "illness",
    ),
]

for source, columns, name in merge_specs:
    master_session = safe_left_merge(
        master_session,
        source,
        columns,
        name,
    )

master_session["injury"] = (
    master_session["injury"]
    .fillna(0)
    .astype(int)
)
master_session["illness"] = (
    master_session["illness"]
    .fillna(0)
    .astype(int)
)

assert not master_session.duplicated(
    ["player_name", "session_id"]
).any()

print("Master athlete-session rows:", len(master_session))
print("Injury-associated athlete-sessions:", int(master_session["injury"].sum()))

print("\nHighest missingness fractions:")
print(
    master_session
    .isna()
    .mean()
    .sort_values(ascending=False)
    .head(15)
)


## 9. Reproducibility and integrity audit

These checks freeze the expected cohort accounting for the current SoccerMon 2020 reconstruction. A failure should be investigated rather than bypassed.


In [ ]:
minute_counts = (
    minute_2020_data
    .groupby(
        ["player_name", "team", "session_id"]
    )
    .agg(
        n_minute_rows=("minute_idx", "size"),
        min_idx=("minute_idx", "min"),
        max_idx=("minute_idx", "max"),
        n_unique_idx=("minute_idx", "nunique"),
    )
    .reset_index()
)

minute_counts["expected_from_index"] = (
    minute_counts["max_idx"]
    - minute_counts["min_idx"]
    + 1
)

minute_counts["has_missing_indices"] = (
    minute_counts["n_unique_idx"]
    != minute_counts["expected_from_index"]
)

cohort_summary = {
    "minute_observations": int(len(minute_2020_data)),
    "team_date_sessions": int(
        minute_2020_data["session_id"].nunique()
    ),
    "athlete_sessions": int(len(master_session)),
    "athletes": int(master_session["player_name"].nunique()),
    "injury_associated_sessions": int(
        master_session["injury"].sum()
    ),
    "positive_athletes": int(
        master_session.loc[
            master_session["injury"] == 1,
            "player_name",
        ].nunique()
    ),
}

EXPECTED_COHORT = {
    "minute_observations": 380_193,
    "team_date_sessions": 251,
    "athlete_sessions": 3_743,
    "athletes": 48,
    "injury_associated_sessions": 22,
    "positive_athletes": 5,
}

assert cohort_summary == EXPECTED_COHORT, (
    "Cohort reconstruction differs from the frozen expected accounting.\n"
    f"Observed: {cohort_summary}\n"
    f"Expected: {EXPECTED_COHORT}"
)

assert not minute_2020_data.duplicated(
    ["player_name", "session_id", "minute"]
).any()

assert not master_session.duplicated(
    ["player_name", "session_id"]
).any()

assert master_session["injury"].isin([0, 1]).all()
assert master_session["illness"].isin([0, 1]).all()

team_a = master_session.loc[
    master_session["team"] == "TeamA"
]
team_b = master_session.loc[
    master_session["team"] == "TeamB"
]

assert len(team_a) == 2_259
assert team_a["player_name"].nunique() == 27
assert int(team_a["injury"].sum()) == 22

assert len(team_b) == 1_484
assert int(team_b["injury"].sum()) == 0

print("=== FROZEN COHORT ACCOUNTING ===")
for key, value in cohort_summary.items():
    print(f"{key}: {value:,}")

print(
    "Athlete-sessions with non-contiguous minute indices:",
    int(minute_counts["has_missing_indices"].sum()),
)

print("Final dataset integrity checks passed.")


## 10. Descriptive temporal audit

The following correlations are descriptive diagnostics only. They are not used as evidence of causality or as a feature-selection procedure.


In [ ]:
audit = master_session[
    [
        "player_name",
        "session_id",
        "srpe_sum",
        "daily_load",
        "weekly_load",
        "atl",
        "ctl28",
        "ctl42",
        "acwr",
        "monotony",
        "strain",
    ]
].copy()

audit["date"] = pd.to_datetime(
    audit["session_id"].str.extract(
        r"(\d{4}-\d{2}-\d{2})"
    )[0],
    errors="coerce",
)

audit = audit.sort_values(
    ["player_name", "date"]
)

audit["srpe_prev"] = (
    audit.groupby("player_name")["srpe_sum"].shift(1)
)
audit["srpe_next"] = (
    audit.groupby("player_name")["srpe_sum"].shift(-1)
)

audit_columns = [
    "srpe_prev",
    "srpe_sum",
    "srpe_next",
    "daily_load",
    "weekly_load",
    "atl",
    "ctl28",
    "ctl42",
    "acwr",
    "monotony",
    "strain",
]

display(
    audit[audit_columns]
    .corr(numeric_only=True)
    .round(3)
)

positive_sessions = (
    master_session.loc[
        master_session["injury"] == 1,
        [
            "player_name",
            "team",
            "session_id",
            "total_minutes",
        ],
    ]
    .sort_values(["player_name", "session_id"])
)

print("\nPositive sessions per athlete:")
print(
    positive_sessions
    .groupby("player_name")
    .size()
    .sort_values(ascending=False)
)


## 11. Export canonical processed tables

In [ ]:
MASTER_SESSION_FILE = OUTPUT_DIR / "master_session_2020.csv"
MINUTE_DATA_FILE = OUTPUT_DIR / "minute_2020_data.csv"
COHORT_SUMMARY_FILE = OUTPUT_DIR / "cohort_summary_2020.csv"

master_session.to_csv(
    MASTER_SESSION_FILE,
    index=False,
)
minute_2020_data.to_csv(
    MINUTE_DATA_FILE,
    index=False,
)
pd.DataFrame(
    [
        {
            "metric": key,
            "value": value,
        }
        for key, value in cohort_summary.items()
    ]
).to_csv(
    COHORT_SUMMARY_FILE,
    index=False,
)

print("Saved:", MASTER_SESSION_FILE)
print("Saved:", MINUTE_DATA_FILE)
print("Saved:", COHORT_SUMMARY_FILE)


## Output contract

A successful run produces:

- `results/processed_data/master_session_2020.csv`
- `results/processed_data/minute_2020_data.csv`
- `results/processed_data/cohort_summary_2020.csv`

The frozen cohort contains 3,743 athlete-sessions from 48 athletes, including 22 injury-associated athlete-sessions from 5 athletes. All positive sessions occur in Team A. These processed outputs are the canonical inputs for downstream modelling notebooks.

The notebook deliberately does not claim minute-specific injury supervision, injury-onset localization, or prospective within-session injury probabilities.
